# 01 · Ingesta Bronze — Transacciones

Carga los archivos de `landing` a la tabla Delta `bronze.transacciones` con **Auto Loader**.

| | |
|---|---|
| **Origen** | `/Volumes/<catalog>/landing/raw_files/transacciones/` |
| **Destino** | `<catalog>.bronze.transacciones` |
| **Idempotencia** | Checkpoint de Auto Loader: cada archivo se ingiere una sola vez |
| **Calidad** | Ningún registro se descarta. Lo que no matchea el schema queda en `_rescued_data` |

**Prerrequisito:** ejecutar los scripts de `setup/` (ver `setup/README.md`).
Decisiones: DT-003 y DT-008 en `docs/TECHNICAL_DECISIONS.md`.

## Parámetros

In [ ]:
dbutils.widgets.text("catalog", "workspace", "Catálogo")
dbutils.widgets.text("source", "transacciones", "Fuente")
dbutils.widgets.dropdown("verify_idempotency", "true", ["true", "false"], "Verificar idempotencia")

CATALOG = dbutils.widgets.get("catalog")
SOURCE = dbutils.widgets.get("source")
VERIFY_IDEMPOTENCY = dbutils.widgets.get("verify_idempotency") == "true"

In [ ]:
import json
import os
import sys
from datetime import datetime, timezone

# La lógica vive en src/ para poder testearla sin cluster (ver tests/).
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

from iol_pipeline.bronze import (
    STATUS_FAIL,
    TRANSACCIONES_COLUMNS,
    ingest_csv_to_bronze,
    validate_ingestion,
)
from iol_pipeline.config import PipelineConfig

spark.conf.set("spark.sql.session.timeZone", "UTC")

cfg = PipelineConfig(catalog=CATALOG)
source_path = cfg.landing_path(SOURCE)
target_table = cfg.bronze_table(SOURCE)
checkpoint_path = cfg.checkpoint_path(SOURCE)

print(f"Origen:     {source_path}")
print(f"Destino:    {target_table}")
print(f"Checkpoint: {checkpoint_path}")

## Precondiciones
Fallar temprano y con un mensaje claro si falta el setup.

In [ ]:
assert spark.catalog.tableExists(target_table), (
    f"No existe la tabla {target_table}. Ejecutar setup/02_bronze.sql"
)

try:
    landing_files = [f for f in dbutils.fs.ls(source_path) if not f.isDir()]
except Exception as e:
    raise AssertionError(
        f"No existe la carpeta {source_path}. Ejecutar setup/01_landing.sql y setup/03_upload_data.sh"
    ) from e

assert landing_files, f"La carpeta {source_path} no tiene archivos"

display(spark.createDataFrame(
    [(f.name, f.size, datetime.fromtimestamp(f.modificationTime / 1000, tz=timezone.utc)) for f in landing_files],
    "archivo STRING, bytes LONG, modificado TIMESTAMP",
))

## Ingesta

In [ ]:
def run_ingestion():
    """Ejecuta una corrida de ingesta y devuelve su resultado validado."""
    ingested_at = datetime.now(timezone.utc).replace(tzinfo=None)
    ingest_csv_to_bronze(
        spark,
        source_path=source_path,
        target_table=target_table,
        checkpoint_path=checkpoint_path,
        columns=TRANSACCIONES_COLUMNS,
        ingested_at=ingested_at,
    )
    return validate_ingestion(spark, target_table, ingested_at)


result = run_ingestion()
display(spark.createDataFrame([result.to_dict()]))

## Verificación de idempotencia
Se re-ejecuta la ingesta sobre los mismos archivos: debe escribir **0 filas**.

In [ ]:
idempotency_ok = None
if VERIFY_IDEMPOTENCY:
    rerun = run_ingestion()
    idempotency_ok = rerun.rows_written == 0
    display(spark.createDataFrame([{**rerun.to_dict(), "idempotente": idempotency_ok}]))
    if not idempotency_ok:
        result.status = STATUS_FAIL
        result.message += f" | Re-ejecución escribió {rerun.rows_written} filas: la ingesta NO es idempotente"
else:
    print("Verificación de idempotencia omitida")

## Estado de la tabla Bronze
Historial de cargas por archivo y corrida: trazabilidad de todo lo que entró.

In [ ]:
display(spark.sql(f"""
    SELECT
        _source_file,
        _ingested_at,
        COUNT(*)              AS filas,
        COUNT(_rescued_data)  AS filas_rescatadas,
        MIN(fecha)            AS fecha_min,
        MAX(fecha)            AS fecha_max
    FROM {target_table}
    GROUP BY _source_file, _ingested_at
    ORDER BY _ingested_at
"""))

## Resultado
Se devuelve al orquestador como JSON.

In [ ]:
summary = {**result.to_dict(), "idempotency_verified": idempotency_ok}
print(json.dumps(summary, indent=2, default=str))
dbutils.notebook.exit(json.dumps(summary, default=str))